# LangChain使用之Chains
## 1、Chains的基本使用
### 1.1 Chain的基本概念
Chain：链，用于将多个组件（提示模板、LLM模型、记忆、工具等）连接起来，形成可复用的 工作流 ，完成复杂的任务。

Chain 的核心思想是通过组合不同的模块化单元，实现比单一组件更强大的功能。比如：
- 将 LLM 与 Prompt Template （提示模板）结合
- 将 LLM 与 输出解析器 结合
- 将 LLM 与 外部数据 结合，例如用于问答
- 将 LLM 与 长期记忆 结合，例如用于聊天历史记录
- 通过将 第一个LLM 的输出作为 第二个LLM 的输入，...，将多个LLM按顺序结合在一起
### 1.2 LCEL 及其基本构成
使用LCEL，可以构造出结构最简单的Chain。

LangChain表达式语言（LCEL，LangChain Expression Language）是一种声明式方法，可以轻松地
将多个组件链接成 AI 工作流。它通过Python原生操作符（如管道符 | ）将组件连接成可执行流程，显
著简化了AI应用的开发。

LCEL的基本构成：提示（Prompt）+ 模型（Model）+ 输出解析器（OutputParser）
即：

```python
# 在这个链条中，用户输入被传递给提示模板，然后提示模板的输出被传递给模型，然后模型的输出被传递给输出解析器。
chain = prompt | model | output_parser
chain.invoke({"input":"What's your name?"})
```
- Prompt：Prompt 是一个 BasePromptTemplate，这意味着它接受一个模板变量的字典并生成一
个 PromptValue 。PromptValue 可以传递给 LLM（它以字符串作为输入）或 ChatModel（它以
消息序列作为输入）。
- Model：将 PromptValue 传递给 model。如果我们的 model 是一个 ChatModel，这意味着它
将输出一个 BaseMessage 。
- OutputParser：将 model 的输出传递给 output_parser，它是一个 BaseOutputParser，意味着
它可以接受字符串或 BaseMessage 作为输入。
- chain：我们可以使用 | 运算符轻松创建这个Chain。 | 运算符在 LangChain 中用于将两个
元素组合在一起。
- invoke：所有LCEL对象都实现了 Runnable 协议，保证一致的调用方式（ invoke / batch / stream ）
### 1.3 Runnable
Runnable是LangChain定义的一个抽象接口（Protocol），它 强制要求 所有LCEL组件实现一组标准方
法：
```
class Runnable(Protocol):
def invoke(self, input: Any) -> Any: ... # 单输入单输出
def batch(self, inputs: List[Any]) -> List[Any]: ... # 批量处理
def stream(self, input: Any) -> Iterator[Any]: ... # 流式输出
# 还有其他方法如 ainvoke（异步）等...
```
任何实现了这些方法的对象都被视为LCEL兼容组件。比如：聊天模型、提示词模板、输出解析器、检索器、代理(智能体)等。
每个 LCEL 对象都实现了 Runnable 接口，该接口定义了一组公共的调用方法。这使得 LCEL 对象链也自动支持这些调用成为可能。
### 1.4 使用举例

In [1]:
# 没有使用chain
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
chat_model = ChatOllama(model = "qwen:7b")
prompt_template = PromptTemplate.from_template(
template = "给我讲一个关于{topic}话题的简短笑话"
)
parser = StrOutputParser()
prompt_value = prompt_template.invoke({"topic":"冰淇淋"})
result = chat_model.invoke(prompt_value)
out_put = parser.invoke(result)
print(out_put)
print(type(out_put))

为什么冰淇淋总是心情不好？
因为它总是“情绪冷却”！
<class 'str'>


In [2]:
# 使用chain
from langchain_core.output_parsers import StrOutputParser
chat_model = ChatOllama(model="qwen:7b")
prompt_template = PromptTemplate.from_template(
template = "给我讲一个关于{topic}话题的简短笑话"
)
parser = StrOutputParser()
# 构建链式调用（LCEL语法）
chain = prompt_template | chat_model | parser
out_put = chain.invoke({"topic": "ice cream"})
print(out_put)
print(type(out_put))

为什么冰淇淋不喜欢去海边？
因为它害怕融化成一滩黏糊糊的甜水。
<class 'str'>


## 2、传统Chain的使用
### 2.1 基础链：LLMChain
#### 2.1.1 使用说明
LCEL之前，最基础也最常见的链类型是LLMChain。

这个链至少包括一个提示词模板（PromptTemplate），一个语言模型（LLM 或聊天模型）。

特点：
- 用于 单次问答，输入一个 Prompt，输出 LLM 的响应。
- 适合 无上下文 的简单任务（如翻译、摘要、分类等）。
- 无记忆：无法自动维护聊天历史
#### 2.1.2 主要步骤
1、配置任务链：使用LLMChain类将任务与提示词结合，形成完整的任务链。
```
chain = LLMChain(llm = llm, prompt = prompt_template)
```
2、执行任务链：使用invoke()等方法执行任务链，并获取生成结果。可以根据需要对输出进行处理和展示。
```
result = chain.invoke(...)
print(result)
```

In [3]:
# 举例1 已经不这么用了
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_classic.chains import LLMChain
# 1、创建大模型实例
chat_model = ChatOllama(model="qwen:7b")
# 2、原始字符串模板
template = "桌上有{number}个苹果，四个桃子和 3 本书，一共有几个水果?"
prompt = PromptTemplate.from_template(template)
# 3、创建LLMChain
llm_chain = LLMChain(
llm=chat_model,
prompt=prompt
)
# 4、调用LLMChain，返回结果
result = llm_chain.invoke({"number": 2})
print(result)

C:\Users\00464841\AppData\Local\Temp\ipykernel_43532\1199811759.py:11: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


{'number': 2, 'text': '桌子上有两种水果：苹果和桃子。\n\n- 苹果有2个。\n- 桃子有4个。\n\n将这两种水果的数量相加：\n\n2（苹果）+ 4（桃子）= 6\n\n所以，一共有6个水果。'}


### 2.2 顺序链之 SimpleSequentialChain
顺序链（SequentialChain）允许将多个链顺序连接起来，每个Chain的输出作为下一个Chain的输入，
形成特定场景的流水线（Pipeline）。
顺序链有两种类型：
- 单个输入/输出：对应着 SimpleSequentialChain
- 多个输入/输出：对应着：SequentialChain
#### 2.2.1 说明
SimpleSequentialChain：最简单的顺序链，多个链 串联执行 ，每个步骤都有 单一 的输入和输出，一
个步骤的输出就是下一个步骤的输入，无需手动映射。

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="qwen:7b")
chainA_template = ChatPromptTemplate.from_messages(
[
("system", "你是一位精通各领域知识的知名教授"),
("human", "请你尽可能详细的解释一下：{knowledge}"),
]
)
chainA_chains = LLMChain(llm=llm,
prompt=chainA_template,
verbose=True
)
chainA_chains.invoke({"knowledge":"什么是LangChain？"})



> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位精通各领域知识的知名教授
Human: 请你尽可能详细的解释一下：什么是LangChain？

> Finished chain.


{'knowledge': '什么是LangChain？',
 'text': '抱歉，您提到的"LangChain"似乎并非一个广泛认可或明确存在于各类学科中的概念。可能这是一个特定领域的术语或者是一个误解，请提供更多上下文信息以便我能给出更准确的解释。'}

In [5]:
from langchain_core.prompts import ChatPromptTemplate
chainB_template = ChatPromptTemplate.from_messages(
[
("system", "你非常善于提取文本中的重要信息，并做出简短的总结"),
("human", "这是针对一个提问的完整的解释说明内容：{description}"),
("human", "请你根据上述说明，尽可能简短的输出重要的结论，请控制在20个字以内"),
]
)
chainB_chains = LLMChain(llm=llm, prompt=chainB_template, verbose=True)

In [6]:
# 导入SimpleSequentialChain
from langchain_classic.chains import SimpleSequentialChain
# 在chains参数中，按顺序传入LLMChain A 和LLMChain B
full_chain = SimpleSequentialChain(chains=[chainA_chains, chainB_chains], verbose=True)
full_chain.invoke({"input":"什么是langChain？"})



> Entering new SimpleSequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位精通各领域知识的知名教授
Human: 请你尽可能详细的解释一下：什么是langChain？

> Finished chain.
"LangChain"似乎不是一个普遍认可或广泛使用的术语。可能是个人发明或者在特定的编程、语言学习或者项目管理环境中提及。

如果"LangChain"确实是一个独特的概念，它可能是指一种语言学习或掌握的序列过程。这个过程中，每个环节（或步骤）都与特定的语言要素相关联。

然而，若要给出一个准确的定义，我需要更具体的信息或上下文来判断"LangChain"的确切含义。


> Entering new LLMChain chain...
Prompt after formatting:
System: 你非常善于提取文本中的重要信息，并做出简短的总结
Human: 这是针对一个提问的完整的解释说明内容："LangChain"似乎不是一个普遍认可或广泛使用的术语。可能是个人发明或者在特定的编程、语言学习或者项目管理环境中提及。

如果"LangChain"确实是一个独特的概念，它可能是指一种语言学习或掌握的序列过程。这个过程中，每个环节（或步骤）都与特定的语言要素相关联。

然而，若要给出一个准确的定义，我需要更具体的信息或上下文来判断"LangChain"的确切含义。
Human: 请你根据上述说明，尽可能简短的输出重要的结论，请控制在20个字以内

> Finished chain.
"LangChain"是个独特的语言学习概念，每个环节关联特定语言要素。

> Finished chain.


{'input': '什么是langChain？', 'output': '"LangChain"是个独特的语言学习概念，每个环节关联特定语言要素。'}

### 2.3 顺序链之 SequentialChain
#### 2.3.1 说明
SequentialChain：更通用的顺序链，具体来说：
- 多变量支持 ：允许不同子链有独立的输入/输出变量。
- 灵活映射 ：需 显式定义 变量如何从一个链传递到下一个链。即精准地命名输入关键字和输出关键字，来明确链之间的关系。
- 复杂流程控制 ：支持分支、条件逻辑（分别通过 input_variables 和 output_variables 配置输入和输出）。

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import SequentialChain
from langchain_ollama import ChatOllama
from langchain_classic.chains import LLMChain
# 创建大模型实例
llm = ChatOllama(model="qwen:7b")
schainA_template = ChatPromptTemplate.from_messages(
[
("system", "你是一位精通各领域知识的知名教授"),
("human", "请你先尽可能详细的解释一下：{knowledge}，并且{action}")
]
)
schainA_chains = LLMChain(llm=llm,
prompt=schainA_template,
verbose=True,
output_key="schainA_chains_key"
)
# schainA_chains.invoke({
# "knowledge": "中国的篮球怎么样？",
# "action": "举一个实际的例子"
# }
# )
schainB_template = ChatPromptTemplate.from_messages(
[
("system", "你非常善于提取文本中的重要信息，并做出简短的总结"),
("human", "这是针对一个提问完整的解释说明内容：{schainA_chains_key}"),
("human", "请你根据上述说明，尽可能简短的输出重要的结论，请控制在100个字以内"),
]
)
schainB_chains = LLMChain(llm=llm,
prompt=schainB_template,
verbose=True,
output_key='schainB_chains_key'
)
Seq_chain = SequentialChain(
chains=[schainA_chains, schainB_chains],
input_variables=["knowledge", "action"],
output_variables=["schainA_chains_key","schainB_chains_key"],
verbose=True)
response = Seq_chain.invoke({
"knowledge":"中国足球为什么踢得烂",
"action":"举一个实际的例子"
}
)
print(response)



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位精通各领域知识的知名教授
Human: 请你先尽可能详细的解释一下：中国足球为什么踢得烂，并且举一个实际的例子

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
System: 你非常善于提取文本中的重要信息，并做出简短的总结
Human: 这是针对一个提问完整的解释说明内容：中国足球之所以在很多国际比赛中表现不佳，原因众多，但以下几个方面可能是关键：

1. **基础训练不足**：相比欧洲等足球强国，中国对足球基础训练的关注度不够，导致球员技术、体能各方面存在短板。

2. **联赛体系不健全**：中国的职业足球联赛（中超）起步较晚，相比之下国内青少年比赛层次不高，难以系统性地培养足球人才。

3. **教练和青训体系不成熟**：中国虽然引进了一些外籍教练，但他们的执教经验和理念在中国的环境中可能不太适用。同时，青训体系中的科学训练、球员心理辅导等环节都相对缺失。

4. **足球文化氛围不够浓厚**：与欧洲国家相比，中国的公众对足球的认知和热情普遍不高，这无疑影响了足球运动在社会层面的发展。

举例来说，2018年俄罗斯世界杯期间，中国国家队小组赛即被淘汰。这不仅是技术上的问题，也是整体青训体系、联赛制度以及文化氛围等方面综合作用的结果。
Human: 请你根据上述说明，尽可能简短的输出重要的结论，请控制在100个字以内

> Finished chain.

> Finished chain.
{'knowledge': '中国足球为什么踢得烂', 'action': '举一个实际的例子', 'schainA_chains_key': '中国足球之所以在很多国际比赛中表现不佳，原因众多，但以下几个方面可能是关键：\n\n1. **基础训练不足**：相比欧洲等足球强国，中国对足球基础训练的关注度不够，导致球员技术、体能各方面存在短板。\n\n2. **联赛体系不健全**：中国的职业足球联赛（中超）起步较晚，相比之下国内青少年比赛层次

#### 3.3.3 顺序链使用场景
场景：多数据源处理
举例：根据产品名

1. 查询数据库获取价格
2. 生成促销文案

In [8]:
from langchain_classic.chains import SequentialChain
from langchain_ollama import ChatOllama
# 创建大模型实例
llm = ChatOllama(model="qwen:7b")
# 第1环节：
query_chain = LLMChain(
llm=llm,
prompt=PromptTemplate.from_template(template="请模拟查询{product}的市场价格，直接返回一个合理的价格数字（如6999），不要包含任何其他文字或代码"),
verbose=True,
output_key="price"
)
# 第2环节：
promo_chain = LLMChain(
llm=llm,
prompt=PromptTemplate.from_template(template="为{product}（售价：{price}元）创作一篇50字以内的促销文案，要求突出产品卖点"),
verbose=True,
output_key="promo_text"
)
sequential_chain = SequentialChain(
chains=[query_chain, promo_chain],
verbose=True,
input_variables=["product"], # 初始输入
output_variables=["price", "promo_text"], # 输出价格和文案
)
result = sequential_chain.invoke({"product": "iPhone16"})
print(result)
# print(result["price"])
# print(result["promo_text"])



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
请模拟查询iPhone16的市场价格，直接返回一个合理的价格数字（如6999），不要包含任何其他文字或代码

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
为iPhone16（售价：7999元）创作一篇50字以内的促销文案，要求突出产品卖点

> Finished chain.

> Finished chain.
{'product': 'iPhone16', 'price': '7999', 'promo_text': '"尊享科技盛宴！全新iPhone16，7999元，搭载顶级处理器，轻薄便携，每一刻都是未来科技的鲜活展示!"'}


### 2.4 数学链 LLMMathChain (了解)
LLMMathChain将用户问题转换为数学问题，然后将数学问题转换为可以使用 Python 的 numexpr 库
执行的表达式。使用运行此代码的输出来回答问题。

In [10]:
from langchain_classic.chains import LLMMathChain
from langchain_ollama import ChatOllama
from langchain_classic.chains import LLMChain
# 创建大模型实例
llm = ChatOllama(model="qwen:7b")
# 创建链
llm_math = LLMMathChain.from_llm(llm)
# 执行链
res = llm_math.invoke("10 ** 3 + 100的结果是多少？")
print(res)

### 2.5 路由链 RouterChain (了解)
路由链（RouterChain）用于创建可以 动态选择下一条链 的链。可以自动分析用户的需求，然后引导到
最适合的链中执行，获取响应并返回最终结果。

比如，我们目前有三类chain，分别对应三种学科的问题解答。我们的输入内容也是与这三种学科对应，
但是随机的，比如第一次输入数学问题、第二次有可能是历史问题... 这时候期待的效果是：可以根据输
入的内容是什么，自动将其应用到对应的子链中。RouterChain就为我们提供了这样一种能力。

### 2.6 文档链 StuffDocumentsChain(了解)
StuffDocumentsChain 是一种文档处理链，它的核心作用是将 多个文档内容合并 （“填充”或“塞
入”）到单个提示（prompt）中，然后传递给语言模型（LLM）进行处理。

使用场景 ：由于所有文档被完整拼接，LLM 能同时看到全部内容，所以适合需要全局理解的任务，如总
结、问答、对比分析等。但注意，仅适合处理 少量/中等长度文档 的场景。

In [ ]:
#1.导入相关包
from langchain_classic.chains import StuffDocumentsChain
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_classic.document_loaders import PyPDFLoader
from langchain_ollama import ChatOllama
# 2.加载PDF
loader = PyPDFLoader("./asset/example/loader.pdf")
#3.定义提示词
prompt_template = """对以下文字做简洁的总结:
{text}
简洁的总结:"""
# 4.定义提示词模版
prompt = PromptTemplate.from_template(prompt_template)
# 5.定义模型
llm = ChatOllama(model="gpt-4o-mini")
# 6.定义LLM链
llm_chain = LLMChain(llm=llm, prompt=prompt )
# 7.定义文档链
stuff_chain = StuffDocumentsChain(
llm_chain=llm_chain,
document_variable_name="text", # 在 prompt 模板中，文档内容应该用哪个变量名表示
) #document_variable_name="text" 告诉 StuffDocumentsChain 把合并后的文档内容填充到 {text}变量中"。
# 8.加载pdf文档
docs = loader.load()
# 9.执行链
res=stuff_chain.invoke(docs)
#print(res)
print(res["output_text"])

## 3、基于LCEL构建的Chains的类型
### 3.1 create_sql_query_chain
create_sql_query_chain，SQL查询链，是创建生成SQL查询的链，用于将 自然语言 转换成 数据库的
SQL查询

In [13]:
from langchain_community.utilities import SQLDatabase
# 连接 MySQL 数据库
db_user = "root"
db_password = "123" #根据自己的密码填写
db_host = "12.0.0.1"
db_port = "3306"
db_name = "tb_name"
# mysql+pymysql://用户名:密码@ip地址:端口号/数据库名
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")
print("哪种数据库：", db.dialect)
print("获取数据表：", db.get_usable_table_names())
# 执行查询
res = db.run("SELECT count(*) FROM employees;")
print("查询结果：", res)

from langchain_ollama import ChatOllama
from langchain_classic.chains.sql_database.query import create_sql_query_chain
# 创建大模型实例
llm = ChatOllama(model="qwen:7b")
# 调用Chain
chain = create_sql_query_chain(llm=llm, db=db)
# response = chain.invoke({"question": "数据表employees中哪个员工工资高？"})
# print(response)
# response = chain.invoke({"question": "查询departments表中一共有多少个部门？"})
# print(response)
# response = chain.invoke({"question": "查询last_name叫King的基本情况"})
# print(response)
# # 限制使用的表
response = chain.invoke({"question": "一共有多少个员工？", "table_names_to_use":
["employees"]})
print(response)